In [ ]:
import transformer_lens
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
import os
from pathlib import Path
import torch
from scipy.stats import pearsonr

SEED = 20250508
RAW_DIR = Path('Data/fig4/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
file = Path(os.environ.get('WORD_LIST_PATH', 'Data/fig4/raw/wiki-100k.txt'))

def load_words():
    with open(file) as word_file:
        valid_words = list(word_file.read().split())
    # remove non letter
    valid_words = [word for word in valid_words if word[:3].isalpha()]
    # remove comments
    valid_words = [word for word in valid_words if not word.startswith('#')]
    # remove urls
    valid_words = [word for word in valid_words if not '.' in word]
    return valid_words

In [ ]:
# load data
data = load_words()
N = len(data)
print('Num data', N)
batch_size = 1024
np.random.seed(SEED)
ind = np.random.choice(N, N, replace=False)
data = [data[i] for i in ind]
print('Examples', data[:5])

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# GPT-2 small (instead of Gemma)
gpt2_model = AutoModelForCausalLM.from_pretrained('gpt2').eval().to(device)
gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

# Pythia-70M
pythia_model = AutoModelForCausalLM.from_pretrained('EleutherAI/pythia-70m').eval().to(device)
pythia_tokenizer = AutoTokenizer.from_pretrained('EleutherAI/pythia-70m')
pythia_tokenizer.pad_token = pythia_tokenizer.eos_token

In [ ]:
gpt2_model

In [ ]:
pythia_model

In [ ]:
# TinyLlama (1.1B)
tinyllama_model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0").eval().to(device)
tinyllama_tokenizer = AutoTokenizer.from_pretrained('TinyLlama/TinyLlama-1.1B-Chat-v1.0')
tinyllama_tokenizer.pad_token = tinyllama_tokenizer.eos_token

In [ ]:
tinyllama_model

In [ ]:
def get_activations(model, tokenizer, texts, layer_idx, device, model_type='gpt2'):
    activations = []

    def hook(module, inp, out):
        while isinstance(out, tuple):
            out = out[0]  # Handles GPT-2 and Pythia
        activations.append(out.detach().cpu())

    if model_type == 'gpt2':
        handle = model.transformer.h[layer_idx].register_forward_hook(hook)
    elif model_type == 'pythia':
        handle = model.gpt_neox.layers[layer_idx].register_forward_hook(hook)
    else:
        raise ValueError(f"Unsupported model_type {model_type}")

    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=10).to(device)

    with torch.no_grad():
        model(**inputs)

    handle.remove()

    return torch.cat(activations, dim=0)

def get_mlp_activations(model, tokenizer, texts, layer_idx, device, model_type='gpt2'):
    activations = []

    def hook(module, inp, out):
        if isinstance(out, tuple):
            out = out[0]  # Ensure output tensor
        activations.append(out.detach().cpu())

    # Register hook on the MLP submodule inside the chosen transformer block
    if model_type == 'gpt2':
        mlp_module = model.transformer.h[layer_idx].mlp
    elif model_type == 'pythia':
        mlp_module = model.gpt_neox.layers[layer_idx].mlp
    else:
        raise ValueError(f"Unsupported model_type {model_type}")

    handle = mlp_module.register_forward_hook(hook)

    # Tokenize inputs
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=10).to(device)

    # Forward pass
    with torch.no_grad():
        model(**inputs)

    handle.remove()

    return torch.cat(activations, dim=0)

def get_gelu_activations(model, tokenizer, texts, layer_idx, device, model_type='gpt2'):
    activations = []

    def hook(module, inp, out):
        activations.append(out.detach().cpu())

    if model_type == 'gpt2':
        gelu_module = model.transformer.h[layer_idx].mlp.act
    elif model_type == 'pythia':
        gelu_module = model.gpt_neox.layers[layer_idx].mlp.act
    elif model_type == 'tinyllama':
        gelu_module = model.model.layers[layer_idx].mlp.act_fn
    else:
        raise ValueError(f"Unsupported model_type {model_type}")

    handle = gelu_module.register_forward_hook(hook)

    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=10)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        model(**inputs)

    handle.remove()

    all_activations = torch.cat(activations, dim=0)
    last_token = inputs['attention_mask'].sum(dim=1).cpu() - 1
    return all_activations[torch.arange(len(texts)), last_token]

In [ ]:
# Gemma activations

num_word = 100

# GPT-2 activations
gpt2_layer_idx = 11
# gpt2_activations = get_activations(
# gpt2_activations = get_mlp_activations(
gpt2_activations = get_gelu_activations(
    gpt2_model, gpt2_tokenizer, data[:num_word], gpt2_layer_idx, device, model_type='gpt2'
)
print("GPT-2 activations shape:", gpt2_activations.shape, gpt2_activations.min())

# Pythia activations (fixed call)
pythia_layer_idx = 5
# pythia_activations = get_activations(
# pythia_activations = get_mlp_activations(
pythia_activations = get_gelu_activations(
    pythia_model, pythia_tokenizer, data[:num_word], pythia_layer_idx, device, model_type='pythia'
)
print("Pythia activations shape:", pythia_activations.shape, pythia_activations.min())

# tinyllama activations (fixed call)
tinyllama_layer_idx = 21
# tinyllama_activations = get_activations(
# tinyllama_activations = get_mlp_activations(
tinyllama_activations = get_gelu_activations(
    tinyllama_model, tinyllama_tokenizer, data[:num_word], tinyllama_layer_idx, device, model_type='tinyllama'
)
print("tinyllama activations shape:", tinyllama_activations.shape, tinyllama_activations.min())

In [ ]:
# Run the model and get logits and activations
gpt2_layer_idx = 11
pythia_layer_idx = 5

activationA, activationB, activationC = [], [], []
for start in tqdm(range(0, N, batch_size)):
    batch = data[start:start + batch_size]
    
    # gpt2_activations = get_activations(
    # gpt2_activations = get_mlp_activations(
    gpt2_activations = get_gelu_activations(
        gpt2_model, gpt2_tokenizer, batch, gpt2_layer_idx, device, model_type='gpt2'
    )
    activationA.append(gpt2_activations.numpy().copy())

    # pythia_activations = get_activations(
    # pythia_activations = get_mlp_activations(
    pythia_activations = get_gelu_activations(
        pythia_model, pythia_tokenizer, batch, pythia_layer_idx, device, model_type='pythia'
    )
    activationB.append(pythia_activations.numpy().copy())

    # tinyllama_activations = get_activations(
    # tinyllama_activations = get_mlp_activations(
    tinyllama_activations = get_gelu_activations(
        tinyllama_model, tinyllama_tokenizer, batch, tinyllama_layer_idx, device, model_type='tinyllama'
    )
    activationC.append(tinyllama_activations.numpy().copy())
activationA = np.concatenate(activationA, 0)
activationB = np.concatenate(activationB, 0)
activationC = np.concatenate(activationC, 0)
N_A, D_A = activationA.shape
N_B, D_B = activationB.shape
N_C, D_C = activationC.shape
assert N_A == N_B
assert N_A == N_C
print('num data', N_A, 'num_neuronA', D_A, 'num_neuronB', D_B, 'num_neuronC', D_C)

In [ ]:
def task(data, y, show_activation=False):
    "Only look at Top9+1 and Bottom9+1, more sampling ToDo later."
    response_ind = np.argsort(-y)
    fig, axes = plt.subplots(3, 7, figsize=(16, 3))
    # positive
    for i in range(3):
        for j in range(3):
            ind = response_ind[i * 3 + j]
            axes[i, j].axis('off')
            title = data[ind]
            if show_activation:
                title += '\n%.2e' % y[ind]
            axes[i, j].set_title(title)
    # negative
    for i in range(3):
        for j in range(3):
            ind = response_ind[::-1][i * 3 + j]
            axes[i, j + 4].axis('off')
            title = data[ind]
            if show_activation:
                title += '\n%.2e' % y[ind]
            axes[i, j + 4].set_title(title)
    # query
    axes[0, 3].axis('off')
    ind = response_ind[9]
    title = data[ind]
    if show_activation:
        title += '\n%.2e' % y[ind]
    axes[0, 3].set_title(title)
    axes[1, 3].axis('off')
    axes[2, 3].axis('off')
    ind = response_ind[::-1][9]
    title = data[ind]
    if show_activation:
        title += '\n%.2e' % y[ind]
    axes[2, 3].set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# check some neurons
np.random.seed(SEED)
for unit_ind in np.random.choice(D_A, 3, replace=False):
    print(unit_ind)
    plt.hist(activationA[:, unit_ind], 100)
    plt.show()
    task(data, activationA[:, unit_ind])

In [ ]:
# check some neurons
np.random.seed(SEED)
for unit_ind in np.random.choice(D_B, 3, replace=False):
    print(unit_ind)
    plt.hist(activationB[:, unit_ind], 100)
    plt.show()
    task(data, activationB[:, unit_ind])

In [ ]:
# check some neurons
np.random.seed(SEED)
for unit_ind in np.random.choice(D_C, 3, replace=False):
    print(unit_ind)
    plt.hist(activationC[:, unit_ind], 100)
    plt.show()
    task(data, activationC[:, unit_ind])

In [ ]:
limit = 10000

In [ ]:
def torch_corr(a, b):
    a = (a - a.mean(0, keepdim=True)) / (torch.linalg.norm(a - a.mean(0, keepdim=True), dim=0, keepdim=True) + 1e-9)
    b = (b - b.mean(0, keepdim=True)) / (torch.linalg.norm(b - b.mean(0, keepdim=True), dim=0, keepdim=True) + 1e-9)
    return a.T @ b

corr = torch_corr(torch.tensor(activationB[:, :limit]), torch.tensor(activationC[:, :limit]))
corr.shape

In [ ]:
plt.imshow(corr, vmin=-1, vmax=1, cmap='bwr')
plt.colorbar()

In [ ]:
plt.hist(corr.flatten(), 100);

In [ ]:
# Load a model as word similarity metric
model_eval = transformer_lens.HookedTransformer.from_pretrained("gpt2-small")

In [ ]:
K = 100
num_test = 10

def compute_mis(y):
    response_ind = np.argsort(-y)
    tmp = [data[i] for i in response_ind[:K + num_test]]
    tmp = tmp + [data[i] for i in response_ind[::-1][:K + num_test]]
    l, a = model_eval.run_with_cache(tmp)
    out = a['ln_final.hook_normalized'][:, -1]
    out = out / torch.linalg.norm(out, dim=1, keepdim=True)
    cossim = (out @ out.T).detach().cpu().numpy()
    tests = []
    for i in range(num_test):
        logit_left = cossim[:K, K + i].mean() - cossim[:K, K + num_test + i].mean()
        logit_right = cossim[-K:, K + i].mean() - cossim[-K:, K + num_test + i].mean()
        tests.append(logit_left - logit_right)
    return tests

In [ ]:
mis_A = []
for i in tqdm(range(min(D_A, limit))):
    mis_A.append(compute_mis(activationA[:, i]))
mis_A = np.array(mis_A)
print(mis_A.shape)

mis_B = []
for i in tqdm(range(min(D_B, limit))):
    mis_B.append(compute_mis(activationB[:, i]))
mis_B = np.array(mis_B)
print(mis_B.shape)

mis_C = []
for i in tqdm(range(min(D_C, limit))):
    mis_C.append(compute_mis(activationC[:, i]))
mis_C = np.array(mis_C)
print(mis_C.shape)

In [ ]:
activationA.dtype

In [ ]:
import pickle

llm_data = {
    'gpt2_activations': activationA.astype(np.float32),
    'pythia_activations': activationB.astype(np.float32),
    'tinyllama_activations': activationC.astype(np.float32),
    'gpt2_mis': mis_A,
    'pythia_mis': mis_B,
    'tinyllama_mis': mis_C,
}

with open(RAW_DIR / 'language-data.pickle', 'wb') as handle:
    pickle.dump(llm_data, handle, protocol=pickle.HIGHEST_PROTOCOL)

# with open(RAW_DIR / 'language-data.pickle', 'rb') as handle:
#     llm_data = pickle.load(handle)

In [ ]:
plt.title('')
stats = []
keys = []
for key in llm_data:
    if 'mis' in key:
        stats.append(llm_data[key].mean(1))
        keys.append(key.split('_')[0])
plt.boxplot(stats)
plt.xticks(np.arange(len(keys)) + 1, keys)
plt.ylabel('MIS')
plt.grid()

In [ ]:
import seaborn as sns

keys = ['gpt2', 'pythia', 'tinyllama']
pretty_keys = {'gpt2': 'GPT2', 'pythia': 'Pythia-70m', 'tinyllama': 'TinyLlama'}
colors = plt.get_cmap('tab10').colors

heatmap = np.zeros((3, 3))
heatmap[np.diag_indices_from(heatmap)] = np.nan

plt.figure(figsize=1 * np.array((9, 3)))
for i, keyA in enumerate(keys):
    avg_mis = llm_data[keyA + '_mis'].mean(1)
    plt.subplot(1, 3, 1 + i)
    for j, keyB in enumerate(keys):
        if i == j:
            continue
        corr = torch_corr(
            torch.tensor(llm_data[keyA + '_activations']),
            torch.tensor(llm_data[keyB + '_activations'])
        )
        max_corrs = abs(corr).max(1)[0]
        plt.scatter(
            avg_mis, max_corrs, 
            s=500 / np.sqrt(len(avg_mis)), 
            alpha=.5, 
            color=colors[j],
            label=pretty_keys[keyB]
        )
        heatmap[i, j] = pearsonr(avg_mis, max_corrs)[0]
    plt.grid()
    plt.title('Source Model: %s' % (pretty_keys[keyA]))
    plt.legend()
    sns.despine(offset=10)
    plt.xlim(np.quantile(avg_mis, .001), np.quantile(avg_mis, .999))
    plt.ylim(np.quantile(max_corrs, .001), np.quantile(max_corrs, .999))
    plt.xlabel('Interpretability [MIS acc.]')
    if i == 0:
        plt.ylabel('Max Matching Unit [corr.]')

plt.tight_layout()
plt.savefig(RAW_DIR / 'language-scatter-raw.pdf')
plt.show()

In [ ]:
import seaborn as sns

keys = ['gpt2', 'pythia', 'tinyllama']
pretty_keys = {'gpt2': 'GPT2', 'pythia': 'Pythia-70m', 'tinyllama': 'TinyLlama'}
colors = plt.get_cmap('tab10').colors

heatmap = np.zeros((3, 3))
heatmap[np.diag_indices_from(heatmap)] = np.nan

plt.figure(figsize=1 * np.array((9, 3)))
for i, keyA in enumerate(keys):
    avg_mis = (llm_data[keyA + '_mis']).mean(1)
    plt.subplot(1, 3, 1 + i)
    for j, keyB in enumerate(keys):
        if i == j:
            continue
        corr = torch_corr(
            torch.tensor(llm_data[keyA + '_activations']),
            torch.tensor(llm_data[keyB + '_activations'])
        )
        max_corrs = abs(corr).max(1)[0]
        plt.scatter(
            avg_mis, max_corrs, 
            s=500 / np.sqrt(len(avg_mis)), 
            alpha=.5, 
            color=colors[j],
            label=pretty_keys[keyB]
        )
        heatmap[i, j] = pearsonr(avg_mis, max_corrs)[0]
    plt.grid()
    plt.title('Source Model: %s' % (pretty_keys[keyA]))
    plt.legend()
    sns.despine(offset=10)
    plt.xlim(np.quantile(avg_mis, .001), np.quantile(avg_mis, .999))
    plt.ylim(np.quantile(max_corrs, .001), np.quantile(max_corrs, .999))
    plt.xlabel('Interpretability [MIS logit]')
    if i == 0:
        plt.ylabel('Max Matching Unit [corr.]')

plt.tight_layout()
plt.savefig(RAW_DIR / 'language-scatter-raw.pdf')
plt.show()

In [ ]:
plt.figure(figsize=0.5 * np.array((8, 6)))
sns.heatmap(
    heatmap, annot=True, fmt=".2f",
    cmap='bwr', vmin=-1, vmax=1,
    cbar_kws={'label': 'Correlation', 'shrink': 0.94},
    annot_kws={"weight": "bold", "fontsize": 10, "color": "black"},
    square=True,
)
plt.xticks(np.arange(3) + .5, pretty_keys.values(), rotation=45)
plt.yticks(np.arange(3) + .5, pretty_keys.values(), rotation=45)
plt.tight_layout()
plt.savefig(RAW_DIR / 'language-correlations-raw.pdf')
plt.show()